# Fetch and Display One DIA Cutout

Fetch a single Science / Template / Difference cutout from the Fink broker
for a chosen `diaObjectId`, overlay cardinal directions (N, E) and the
DCR-expected direction (Zenith), and draw the **dipole arrow** derived from
`r:dipoleAngle` using the Rubin convention
`dipole_PA_deg = (90 − r:dipoleAngle) mod 360`.


- **Author :** Sylvie Dagoret-Campagne
- **Creation date :** 2026-06-12
- **Last update :** 2026-06-13
- **Affiliation :** IJCLab / IN2P3 / CNRS – Université Paris-Saclay
- **Context :** Rubin/LSST sky-alert dipole artifact analysis


In [ ]:
# Standard library
import io
import os

# HTTP client
import requests

# Data wrangling
import pandas as pd
import numpy as np

# Plotting
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize

# Scipy utilities (kept for potential future use)
from scipy.ndimage import maximum_filter, label, rotate
from scipy.optimize import curve_fit
from scipy.interpolate import griddata

# Astropy – FITS, WCS, coordinates, visualisation
from astropy.io import fits
from astropy.wcs import WCS
from astropy.coordinates import SkyCoord, AltAz, EarthLocation, SkyOffsetFrame
from astropy.time import Time
import astropy.units as u
from astropy.visualization import ZScaleInterval
from astropy.visualization.mpl_normalize import ImageNormalize

In [ ]:
# ---------------------------------------------------------------------------
# Rubin Observatory site constants (ITRF coordinates)
# ---------------------------------------------------------------------------
RUBIN_LAT_DEG = -30.244728  # geodetic latitude  [deg]
RUBIN_LON_DEG = -70.749417  # east longitude     [deg]
RUBIN_HEIGHT_M = 2647.0  # elevation above sea level [m]

In [ ]:
def parallactic_angle(obs_time, ra, dec, location):
    """Compute the parallactic angle q at (ra, dec) for a given observation.

    Parameters
    ----------
    obs_time : astropy.time.Time
    ra, dec  : astropy.units.Quantity   (angle, in radians internally)
    location : astropy.coordinates.EarthLocation

    Returns
    -------
    q : astropy.units.Quantity  (angle, radians)
        Positive when zenith is to the east of north.
    """
    target = SkyCoord(ra=ra, dec=dec, unit=u.rad)

    # Local sidereal time → hour angle
    lst = obs_time.sidereal_time("apparent", longitude=location.lon)
    ha = (lst - target.ra).to(u.rad)

    lat = location.lat.to(u.rad)
    dec = target.dec.to(u.rad)

    # Standard formula (Meeus, Astronomical Algorithms)
    q = np.arctan2(np.sin(ha), np.tan(lat) * np.cos(dec) - np.sin(dec) * np.cos(ha))
    return q

## WCS pixel → sky transformation

The FITS WCS linear transform relates pixel offsets to sky offsets via the PC matrix:

$$\begin{pmatrix} \Delta \alpha \\ \Delta \delta \end{pmatrix} = \mathbf{PC} \cdot \begin{pmatrix} x - CRPIX1 \\ y - CRPIX2 \end{pmatrix}$$

Directions (North, East, Zenith, Dipole) are projected into pixel space
using `WCS.world_to_pixel_values` applied to small sky offsets from the image centre.


In [ ]:
def plot_cutout_wcs_with_directions(
    fits_file,
    dipole_angle_rubin=None,
    dipole_length_pix=None,
    cmap="viridis",
    zscale=True,
):
    """Display a FITS cutout with overlaid direction arrows and optional dipole arrow.

    Draws:
      - Red    arrow : celestial North
      - Blue   arrow : celestial East
      - Orange arrow : direction toward Zenith (DCR shift direction)
      - Cyan   double-headed arrow : dipole axis (if dipole_angle_rubin is given)

    Parameters
    ----------
    fits_file : str
        Path to a FITS file that contains WCS, MJD-OBS, and observatory keywords.
    dipole_angle_rubin : float or None
        Value of ``r:dipoleAngle`` from the Fink alert (degrees, Rubin convention).
        The position angle shown is ``dipole_PA_deg = (90 - dipole_angle_rubin) % 360``.
    dipole_length_pix : float or None
        Total dipole length in pixels to use for the arrow half-length.
        Defaults to 20 % of the image half-width when None.
    cmap : str
        Matplotlib colormap name (default: 'viridis').
    zscale : bool
        Apply ZScale stretch to the image (default: True).
    """
    # ------------------------------------------------------------------
    # Load FITS data and header
    # ------------------------------------------------------------------
    with fits.open(fits_file) as hdul:
        data = hdul[0].data.copy()
        hdr = hdul[0].header.copy()

    wcs = WCS(hdr)

    # ------------------------------------------------------------------
    # Observation time (robust fallback to TAI)
    # ------------------------------------------------------------------
    timesys = str(hdr.get("TIMESYS", "tai")).lower()
    obstime = Time(hdr["MJD-OBS"], format="mjd", scale=timesys)

    # ------------------------------------------------------------------
    # Observatory location (use header keywords, fall back to Rubin constants)
    # ------------------------------------------------------------------
    loc = EarthLocation(
        lat=hdr.get("OBS-LAT", RUBIN_LAT_DEG) * u.deg,
        lon=hdr.get("OBS-LONG", RUBIN_LON_DEG) * u.deg,
        height=hdr.get("OBS-ELEV", RUBIN_HEIGHT_M) * u.m,
    )

    # Telescope rotator angle (may be missing)
    rotpa = hdr.get("ROTPA", np.nan) * u.deg

    # ------------------------------------------------------------------
    # Image geometry: centre pixel coordinates
    # ------------------------------------------------------------------
    ny, nx = data.shape
    x0, y0 = nx / 2.0, ny / 2.0

    # Sky coordinates of the image centre
    ra0, dec0 = wcs.pixel_to_world_values(x0, y0)
    sky_center = SkyCoord(ra=ra0 * u.deg, dec=dec0 * u.deg, frame="icrs")

    # ------------------------------------------------------------------
    # Local tangent-plane frame (SkyOffsetFrame) at image centre
    # East is the longitude axis, North is the latitude axis.
    # ------------------------------------------------------------------
    local_frame = SkyOffsetFrame(origin=sky_center)

    # Cardinal probes: +10 arcsec North and East in the local frame
    north_sky = SkyCoord(0 * u.arcsec, +10 * u.arcsec, frame=local_frame).icrs
    east_sky = SkyCoord(+10 * u.arcsec, 0 * u.arcsec, frame=local_frame).icrs

    # ------------------------------------------------------------------
    # Altitude / azimuth and parallactic angle at the image centre
    # ------------------------------------------------------------------
    altaz = sky_center.transform_to(AltAz(obstime=obstime, location=loc))
    alt = altaz.alt
    az = altaz.az
    zenith_dist = 90 * u.deg - alt
    q_rad = parallactic_angle(obstime, sky_center.ra, sky_center.dec, loc).to_value(u.rad)
    q_deg = np.degrees(q_rad)

    print(f"Sky center        : RA={ra0:.5f} deg, Dec={dec0:.5f} deg")
    print(f"Altitude          : {alt:.3f}")
    print(f"Zenith distance   : {zenith_dist:.3f}")
    print(f"Azimuth           : {az:.3f}")
    print(f"Parallactic angle : {q_deg:.3f} deg")
    print(f"ROTPA             : {rotpa}")

    # ------------------------------------------------------------------
    # Project sky probes into pixel-offset vectors
    # ------------------------------------------------------------------
    def world_to_vec(coord):
        """Return pixel offset (dx, dy) from image centre for a sky coordinate."""
        x, y = wcs.world_to_pixel_values(coord.ra.deg, coord.dec.deg)
        return x - x0, y - y0

    vx_n, vy_n = world_to_vec(north_sky)
    vx_e, vy_e = world_to_vec(east_sky)

    # Zenith direction: rotate North toward East by the parallactic angle q
    # (the zenith lies at angle q from North in the local tangent plane)
    vx_z = np.cos(q_rad) * vx_n + np.sin(q_rad) * vx_e
    vy_z = np.cos(q_rad) * vy_n + np.sin(q_rad) * vy_e

    # ------------------------------------------------------------------
    # Normalisation scale: arrows will be arrow_frac * half-image wide
    # ------------------------------------------------------------------
    arrow_frac = 0.22  # fraction of image half-size
    scale = arrow_frac * min(nx, ny) / 2.0
    # Unit pixel vectors for North, East, Zenith
    mag_n = np.hypot(vx_n, vy_n)
    mag_e = np.hypot(vx_e, vy_e)
    mag_z = np.hypot(vx_z, vy_z)

    dx_n, dy_n = vx_n / mag_n * scale, vy_n / mag_n * scale
    dx_e, dy_e = vx_e / mag_e * scale, vy_e / mag_e * scale
    dx_z, dy_z = vx_z / mag_z * scale, vy_z / mag_z * scale

    aw = max(0.3, scale * 0.04)  # arrow shaft width  (proportional to scale)
    hw = aw * 3  # arrow head width
    hl = aw * 4  # arrow head length

    # ------------------------------------------------------------------
    # Dipole arrow (double-headed, cyan)
    # Convention: dipole_PA_deg = (90 - r:dipoleAngle) % 360
    # PA is measured East of North in the WCS tangent plane.
    # ------------------------------------------------------------------
    has_dipole = dipole_angle_rubin is not None
    if has_dipole:
        dipole_pa_deg = (90.0 - dipole_angle_rubin) % 360.0
        dipole_pa_rad = np.radians(dipole_pa_deg)
        # Dipole pixel vector (unit vector in the N/E basis, rotated by PA)
        # PA = 0  → North, PA = 90 → East  (standard astronomical convention)
        vx_dip = np.sin(dipole_pa_rad) * vx_e / mag_e + np.cos(dipole_pa_rad) * vx_n / mag_n
        vy_dip = np.sin(dipole_pa_rad) * vy_e / mag_e + np.cos(dipole_pa_rad) * vy_n / mag_n
        # Half-length of the dipole arrow in pixels
        if dipole_length_pix is not None:
            half_len = dipole_length_pix / 2.0
        else:
            half_len = scale * 0.9  # default: slightly shorter than the cardinal arrows

    # ------------------------------------------------------------------
    # Image stretch (ZScale or raw)
    # ------------------------------------------------------------------
    if zscale:
        interval = ZScaleInterval()
        vmin, vmax = interval.get_limits(data)
        norm = Normalize(vmin=vmin, vmax=vmax)
    else:
        norm = None

    # ------------------------------------------------------------------
    # Figure
    # ------------------------------------------------------------------
    fig = plt.figure(figsize=(7, 7))
    ax = plt.subplot(projection=wcs)

    ax.imshow(data, origin="lower", cmap=cmap, norm=norm)
    ax.coords.grid(True, color="white", ls="dotted", lw=0.8, alpha=0.6)
    ax.set_xlabel("Right Ascension", fontsize=11)
    ax.set_ylabel("Declination", fontsize=11)

    # Cardinal and zenith arrows – drawn in pixel coordinates
    kw = dict(
        width=aw,
        head_width=hw,
        head_length=hl,
        length_includes_head=True,
        transform=ax.get_transform("pixel"),
    )
    ax.arrow(x0, y0, dx_n, dy_n, color="tomato", **kw)
    ax.arrow(x0, y0, dx_e, dy_e, color="dodgerblue", **kw)
    ax.arrow(x0, y0, dx_z, dy_z, color="darkorange", **kw)

    # Arrow labels placed slightly beyond each arrow tip
    label_offset = 1.18
    ax.text(
        x0 + dx_n * label_offset,
        y0 + dy_n * label_offset,
        "N",
        color="tomato",
        fontsize=11,
        fontweight="bold",
        ha="center",
        va="center",
        transform=ax.get_transform("pixel"),
    )
    ax.text(
        x0 + dx_e * label_offset,
        y0 + dy_e * label_offset,
        "E",
        color="dodgerblue",
        fontsize=11,
        fontweight="bold",
        ha="center",
        va="center",
        transform=ax.get_transform("pixel"),
    )
    ax.text(
        x0 + dx_z * label_offset,
        y0 + dy_z * label_offset,
        "Z",
        color="darkorange",
        fontsize=11,
        fontweight="bold",
        ha="center",
        va="center",
        transform=ax.get_transform("pixel"),
    )

    # Dipole: double-headed arrow (positive and negative lobes)
    if has_dipole:
        kw_dip = dict(
            width=aw * 1.2,
            head_width=hw * 1.3,
            head_length=hl,
            length_includes_head=True,
            transform=ax.get_transform("pixel"),
            color="cyan",
        )
        ax.arrow(x0, y0, vx_dip * half_len, vy_dip * half_len, **kw_dip)  # forward lobe
        ax.arrow(x0, y0, -vx_dip * half_len, -vy_dip * half_len, **kw_dip)  # backward lobe

        ax.text(
            x0 + vx_dip * half_len * 1.25,
            y0 + vy_dip * half_len * 1.25 - 2,
            f"Dipole\nPA={dipole_pa_deg:.1f}°",
            color="cyan",
            fontsize=9,
            fontweight="bold",
            ha="center",
            va="center",
            transform=ax.get_transform("pixel"),
        )

    # Legend
    patches = [
        mpatches.Patch(color="tomato", label="North"),
        mpatches.Patch(color="dodgerblue", label="East"),
        mpatches.Patch(color="darkorange", label=f"Zenith  (q={q_deg:.1f}°)"),
    ]
    if has_dipole:
        patches.append(
            mpatches.Patch(color="cyan", label=f"Dipole  (r:dipoleAngle={dipole_angle_rubin:.1f}°)")
        )
    ax.legend(
        handles=patches, loc="lower right", fontsize=8, framealpha=0.6, facecolor="k", labelcolor="white"
    )

    # Informative multi-line title
    fname_base = os.path.basename(fits_file)
    title_lines = [
        fname_base,
        (
            f"MJD={hdr.get('MJD-OBS', float('nan')):.4f}  "
            f"Alt={alt.to_value(u.deg):.1f}°  "
            f"z={zenith_dist.to_value(u.deg):.1f}°  "
            f"q={q_deg:.1f}°"
        ),
    ]
    ax.set_title("\n".join(title_lines), fontsize=9, pad=8)

    plt.tight_layout()
    plt.show()

## Object selection

Dictionary of COSMOS Deep Drilling Field diaObjects ranked by dipole fraction.
Change `DIAOBJECT_IDX` to switch between objects.


In [ ]:
# Candidate diaObjects in the COSMOS DDF, ranked by dipole fraction
objsid = {
    # 0: 313888627167330394,   # rank 1  – COSMOS, dipole_frac=0.962, n_dipoles=507, mostly before 2024-03
    0: 313985344866353157,  # rank 2  – COSMOS, positive & negative fluxes, small dipoles (many bands)
    1: 313853517840777344,  # rank 3  – COSMOS, positive & negative fluxes, small dipoles
    2: 313972182542712999,  # rank 4  – COSMOS, positive & negative fluxes, range of dipole lengths
    3: 313871013109563545,  # rank 5  – COSMOS, positive & negative fluxes, range of dipole lengths
    4: 313871013420466334,  # rank 6  – COSMOS, positive & negative fluxes, mixed dipole sizes
    5: 313998569477505082,  # rank 7  – COSMOS, positive & negative fluxes
    6: 313994141002367046,  # rank 8  – COSMOS, QSO, two dipole populations in g band
    7: 313888627167330394,  # rank 1  – same as commented rank-1 above
}

In [ ]:
DIAOBJECT_IDX = 3  # ← change this index to select a different object
DIAOBJECT_ID = objsid[DIAOBJECT_IDX]
print(f"Selected diaObjectId: {DIAOBJECT_ID}  (index {DIAOBJECT_IDX})")

## 1 – Fetch the diaSource table from Fink

Retrieve all diaSources associated to the selected diaObject, including
flux columns and dipole characterisation columns (`r:dipoleAngle`, etc.).


In [ ]:
diaObjectId = str(DIAOBJECT_ID)

# Request all relevant source columns plus dipole characterisation
r = requests.post(
    "https://api.lsst.fink-portal.org/api/v1/sources",
    json={
        "diaObjectId": diaObjectId,
        "columns": (
            "r:diaSourceId, r:midpointMjdTai, r:ra, r:raErr, r:dec, r:decErr, "
            "r:apFlux, r:apFluxErr, r:scienceFlux, r:scienceFluxErr, "
            "r:templateFlux, r:templateFluxErr, r:band, "
            "r:dipoleAngle, r:dipoleLength, r:isDipole"
        ),
    },
)

if r.status_code == 200:
    data_sources = pd.read_json(io.BytesIO(r.content))
else:
    raise RuntimeError(f"Fink API error for diaObjectId {diaObjectId}: {r.status_code} – {r.text}")

data_sources.to_csv(f"data_{diaObjectId}.csv", index=False)
display(data_sources.head(10))

In [ ]:
# Quick sanity-check: scienceFlux light curve, one colour per photometric band
fig, ax = plt.subplots(figsize=(11, 4))
for band, grp in data_sources.groupby("r:band"):
    ax.errorbar(
        grp["r:midpointMjdTai"],
        grp["r:scienceFlux"],
        yerr=grp["r:scienceFluxErr"],
        fmt="o",
        ms=4,
        label=band,
        alpha=0.7,
    )
ax.axhline(0, color="k", lw=0.8, ls="--")
ax.set_xlabel("MJD (TAI)", fontsize=11)
ax.set_ylabel("scienceFlux [nJy]", fontsize=11)
ax.set_title(f"Light curve – diaObjectId: {diaObjectId}", fontsize=12)
ax.legend(title="band", fontsize=8)
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 2 – Download FITS cutouts for the first diaSource

For the first entry in the source table, download the Science, Template
and Difference cutouts from Fink, inject the missing time/location
keywords, and save to FITS files on disk.


In [ ]:
for row_idx in range(len(data_sources["r:diaSourceId"])):
    src = str(data_sources["r:diaSourceId"][row_idx])
    mjd_val = data_sources["r:midpointMjdTai"][row_idx]

    obstime_row = Time(mjd_val, format="mjd", scale="tai")
    mjd_str = str(mjd_val).replace(".", "_")

    print(f"Processing diaSourceId: {src}   MJD: {mjd_val:.6f}")

    for kind in ["Science", "Template", "Difference"]:
        r_cut = requests.post(
            "https://api.lsst.fink-portal.org/api/v1/cutouts",
            json={"diaSourceId": src, "kind": kind, "output-format": "FITS"},
        )

        if r_cut.status_code == 200 and len(r_cut.content) > 0:
            try:
                with fits.open(io.BytesIO(r_cut.content), ignore_missing_simple=True) as data_cut:
                    hdr = data_cut[0].header

                    # Inject observation time (needed for parallactic angle computation)
                    hdr["MJD-OBS"] = (mjd_val, "Observation midpoint [MJD, TAI]")
                    hdr["TIMESYS"] = ("TAI", "Time system")
                    hdr["DATE-OBS"] = (obstime_row.utc.isot, "UTC ISO observation time")
                    hdr["COMMENT"] = "Time keywords injected from Fink midpointMjdTai"

                    # Inject Rubin Observatory site coordinates
                    hdr["OBS-LAT"] = (RUBIN_LAT_DEG, "Rubin latitude  [deg]")
                    hdr["OBS-LONG"] = (RUBIN_LON_DEG, "Rubin longitude [deg]")
                    hdr["OBS-ELEV"] = (RUBIN_HEIGHT_M, "Rubin elevation [m]")

                    filename = f"{mjd_str}_cutout_{kind}.fits"
                    data_cut.writeto(filename, overwrite=True)
                    print(f"  Saved {filename}")

            except Exception as exc:
                print(f"  ERROR processing diaSourceId={src} kind={kind}: {exc}")
        else:
            print(f"  No content for diaSourceId={src} kind={kind} (HTTP {r_cut.status_code})")

    break  # process only the first diaSource

In [ ]:
# Build the FITS file paths for the three cutout types
selected_prefix = f"{mjd_str}_cutout"
file_sci = f"{selected_prefix}_Science.fits"
file_temp = f"{selected_prefix}_Template.fits"
file_diff = f"{selected_prefix}_Difference.fits"
print("Science   :", file_sci)
print("Template  :", file_temp)
print("Difference:", file_diff)

## 3 – Inspect the Difference FITS header

Print and display the FITS header of the Difference cutout to verify that
the WCS and time keywords are correctly set.


In [ ]:
# Read and display the Difference cutout header
with fits.open(file_diff) as hdu:
    header_diff = hdu[0].header
    img_diff = hdu[0].data

display(header_diff)

In [ ]:
# Print the WCS object to confirm pixel scale and orientation
print(WCS(header_diff))

In [ ]:
# Raw preview of the Difference image (no stretch, pixel coordinates)
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(img_diff, origin="lower", cmap="viridis")
plt.colorbar(im, ax=ax, label="Pixel value")
ax.set_title("Difference cutout – raw pixel values")
plt.tight_layout()
plt.show()

## 4 – Extract dipole parameters for the first diaSource

Retrieve `r:dipoleAngle` and `r:dipoleLength` for the source whose cutout
was downloaded, then pass them to the visualisation function.


In [ ]:
# Use the first row (same source for which the cutouts were downloaded)
first_row = data_sources.iloc[0]

dipole_angle_rubin = None
dipole_length_pix = None

if "r:dipoleAngle" in first_row and pd.notna(first_row["r:dipoleAngle"]):
    dipole_angle_rubin = float(first_row["r:dipoleAngle"])
    # dipole_pa_deg      = (90.0 - dipole_angle_rubin) % 360.0
    dipole_pa_deg = (dipole_angle_rubin) % 360.0
    print(f"r:dipoleAngle  = {dipole_angle_rubin:.2f} deg  (Rubin convention)")
    print(f"dipole_PA_deg  = {dipole_pa_deg:.2f} deg  (East of North)")
else:
    print("r:dipoleAngle not available for this source – no dipole arrow will be drawn.")

if "r:dipoleLength" in first_row and pd.notna(first_row["r:dipoleLength"]):
    dipole_length_pix = float(first_row["r:dipoleLength"])
    print(f"r:dipoleLength = {dipole_length_pix:.2f} pixels")

## 5 – Display Difference cutout with WCS directions and dipole arrow


In [ ]:
# Display the Difference cutout with all direction overlays
plot_cutout_wcs_with_directions(
    file_diff,
    dipole_angle_rubin=dipole_angle_rubin,
    dipole_length_pix=dipole_length_pix,
    cmap="viridis",
    zscale=True,
)

## 6 – Science and Template cutouts (for reference)


In [ ]:
# Display Science and Template side by side for comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, fname, label_str in zip(axes, [file_sci, file_temp], ["Science", "Template"]):
    with fits.open(fname) as hdu:
        img = hdu[0].data
    interval = ZScaleInterval()
    vmin, vmax = interval.get_limits(img)
    im = ax.imshow(img, origin="lower", cmap="viridis", vmin=vmin, vmax=vmax)
    plt.colorbar(im, ax=ax, label="Pixel value")
    ax.set_title(f"{label_str} – MJD {mjd_val:.4f}", fontsize=10)
    ax.set_xlabel("x [pix]")
    ax.set_ylabel("y [pix]")

plt.suptitle(f"diaObjectId: {diaObjectId}", fontsize=12, y=1.01)
plt.tight_layout()
plt.show()